In [4]:
#imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import scipy.optimize
import sympy as sp
import pylab as py
import decimal as dc


from numpy import random
from scipy.optimize import curve_fit
from scipy.special import factorial
from scipy.stats import poisson
from scipy.integrate import quad
from scipy import signal
from IPython.display import display, Math, Latex
from sympy import separatevars
from decimal import Decimal, ROUND_HALF_UP

os.chdir(r'C:\Users\lhafn\Documents\Uni\PAP2')

# Daten einlesen

Dateien einlesen (als Numpy Arrays und Pandas Dataframes)

In [5]:
#Read date from a .txt file. usecols = (0,1,2,...) might be usefull
data_nd = np.genfromtxt("testdaten.txt",skip_header=2)
#a n-dimesional numpy array is created
#print(data_nd[0,0])

#If names or datatypes for specific columns are specified one gets a 1D array of n-tuples
data = np.genfromtxt("testdaten.txt",dtype = "int,float,float",skip_header=2,names = ["Zeit","Messwert1","Messwert2"])
#print(data[0][0])

#Conversion to 2d array
array = np.array(data.tolist())

#Conversion to pandas Dataframes
df = pd.DataFrame(data)

#And back to numpy
data_new = df.to_numpy(dtype = "float")
#This conversion is not as flexible as names get lost and the common datatype of the frame is used

# Funktionen

In [6]:
#     Autor: Daniel Härterich #verändert
"""
    Funktion formatiert Arrays an Daten in Latex-Tabellenformat
    data: Daten als 2D-Array[[x1,x2,..],[a1,a2,...],...]
    header: Spaltenüberschriften als 1D-Array mit gleicher Zeilenlänge wie Daten-Array
    caption: optionaler string als Titel der Tabelle

    Hinweis: Latex Namen mit doppeltem \\ 
"""

def latex_tabelle(data, header=None, caption=''): 
    format = '|c'
    for a in range(np.size(header)-1):
        #print(format)
        format += '|c'
    format +="|"
    tex = ('\\begin{table}\\label{}\n\t\\centering\n\t\\caption{'
           +caption+ '}\n\t\\begin{tabular}{' + format +'}\n \\hline\t\t')
    if len(header)>0:
        head = str(header[0])
        for i in range(1, np.size(header)):
            head += ' & ' + str(header[i])
        head += ' \\\\ ' + '\\hline' 
        tex += head
    
    if len(data)>0:
        for j in range(np.size(data[0,:])):
            zeile = '\n\t\t' + str(data[0][j])
            for i in range(1, np.size(data[:,j])):
                zeile += ' & ' + str(data[i][j])
            zeile += ' \\\\'
            tex += zeile
    tex += '\n\\hline\n\t\\end{tabular}\n\\end{table}'
    print(tex)

"""
Wandelt einen einzelnen Wert in einen Array der Länge eines anderen Arrays um.
Oder expandiert einen Arrax, in dem er jede Stelle mehrfach nimmt
a: Input, zahl oder 1D Array
reference: Array, an den die Länge angepasst werden soll
repeat: Anzahl wie oft jeder Wert des Arrays im neuen Array auftauchen soll 

Hinweis: repeat*len(a) sollte der Länge von allen anderen Arrays der Tabelle später entsprechen, sonst kommt es zu Problemen
"""
def array_anpassung(a, reference=None, repeat=1):
    x = np.array([])
    if np.ndim(a)!= 0:
        for j in range(np.size(a)):
            for k in range(repeat):
                x = np.append(x, a[j])
    else:
        for i in range(np.size(reference)):
            x = np.append(x, a)
    return x

""" 
Generiert einen Array der Länge eines Referenzarrays mit Zahlen 1,2,...
"""
def nummer_array_generator(reference):
    r = np.array([])
    for i in range(np.size(reference)):
        r = np.append(r, (i+1))
    return r

In [7]:
#     Autor: Daniel Härterich
"""
Rundet Daten oder ihren Fehler auf signifikante Stellen. Entweder nur Fehler angeben um Fehler zu runden oder Fehler und Daten angeben um Daten auf signifikante Stellen passend zum Fehler gerundet zu erhalten.

daten: 1D array mit gleicher Länge wie Fehler, Zahl, oder nichts
fehler: 1D array, oder Zahl

Problem, das nicht gefixt ist/ wird: Signifikante 0-Stellen am Ende werden nicht richtig angezeigt. Hier manuell nachbessern!
"""
def significant(fehler, daten=np.array([])):
    x=0
    x_a=np.array([])
    #print("1",fehler)
    if np.isscalar(fehler):
        x = signifikante_Stellen(fehler)
    else:
        for i in range(np.size(fehler)):
            #print("2",i,fehler)
            x_a = np.append(x_a, signifikante_Stellen(fehler[i]))
    if np.isscalar(daten):
        return korrekt_Runden(daten, x)
    elif np.size(daten)!=0:
        daten_round = np.zeros(len(daten))
        if np.isscalar(fehler):
            for j in range(np.size(daten)):
                daten_round[j] = korrekt_Runden(daten[j], x)
        else:
            for k in range(np.size(daten)):
                daten_round[k] = korrekt_Runden(daten[k], x_a[k])
        return daten_round
    else:
        if np.isscalar(fehler):
            return korrekt_Runden(fehler, x)
        else:
            fehler_round = np.zeros(len(fehler))
            for l in range(np.size(fehler)):
                fehler_round[l] = korrekt_Runden(fehler[l], x_a[l])
            return fehler_round
        
# Minifunktion um die Signifikanten Stellen herauszufinden
def signifikante_Stellen(fehler):
    if np.ndim(fehler) == 0:
        x = int(np.floor(np.log10(fehler)))
        sigf = Decimal(fehler/10**(x))
        if(sigf<3):
            x-=1
        return x
    else:
        x = np.floor(np.log10(fehler))
        sigf = fehler/10**(x)
        for i in range(np.size(x)):
            if(sigf[i]<3):
                x[i]-=1
        return x

# Minifunktion um Zahl an bestimmter Stelle abzuschneiden und korrekt zu runden
def korrekt_Runden(Zahl, Stelle):
    ten = Decimal(10)
    ret = Decimal(Zahl)/ten**(Decimal(Stelle))+Decimal('0.01')
    ret = ret.quantize(dc.Decimal('1'),dc.ROUND_HALF_UP)
    ret=ret*ten**(Decimal(Stelle))
    return ret

In [8]:
"""
    Bringt einen Wert mit Fehler in das Format für eine Latex Tabelle

    Value: Datenwert als Zahl oder 1D array
    error: Fehler als Zahl oder 1D array, bei konstantem Fehler über einen 1D Array für Value reicht es eine Zahl anzugeben.
    significant: Rundet auf signifikante Stellen nach Daniels Funktion
"""
def to_latex(value,error,sig = True):
    if sig:
        value = significant(error,daten = value)
        error = significant(error)
    if(np.isscalar(value)):
        latex = str("$" + str(value) +  chr(92) +"pm " + str(error)+"$")
    else:
        if(np.isscalar(error)):
            error = np.full(len(value),error)
        latex = np.empty(len(value),dtype = object)
        for i in range(len(value)):
            latex[i] = str("$" + str(value[i]) +  chr(92) +"pm " + str(error[i])+"$")
    return latex

In [9]:
#Funktion für Sigma-Abweichung
def Abweichung(x, fx, y, fy):
    return abs(x-y)/ np.sqrt(fx**2 +fy**2)

#Funktion für prozentuale Abweichung
def Abweichung_proz(x, y):
    return 100*abs(x-y)/ (x)

In [10]:

"""
errf gibt zu einer Funktion und deren fehlerbehafteten Parametern den Fehler nach Gaußscher Fehlerfortpflanzung aus
function: eine sympy Funktion
param: alle fehlerbehafteten Parameter in einem Array
"""
def errf(function, param):
    error = 0
    parameters = []
    for par in param:
        d = sp.symbols('dstringdenniemandbenutzt' + par.name)
        partial = sp.diff(function, par) * d    
        error = error + partial**2
        parameters.append((par,d))
    error_abs=sp.simplify(sp.sqrt(error),rational = True)             
    error_rel=sp.simplify(sp.sqrt(error/function**2),rational = True)
    return(error_abs, error_rel, parameters)

In [11]:
"""
Berechnet Wert und Fehler für Daten
f: eine sympy Funktion
err: eine sympy Funktion, die den Fehler berechnet
data: Als array von Tupeln der Form [(a,da),(b,db),...] oder als array/liste [a,da,b,db,...)] (Reihenfolge muss die sein, in der die Argumente in
    der Funktion genommen werden)
"""
def evaluate_with_error(f, err, values):
    if (np.ndim(values) != 1):                    #Falls der Input in mehrere Tupel aufgeteilt ist, werden diese zu einem Array zusammengefügt 
        values = np.concatenate(values)
    #print(values)
    result = f(*values[::2])
    uncertainty = err(*values)
    return result, uncertainty
    

In [12]:
"""
Darstellung einer Funktion in Latex
f: eine sympy Funktion
"""
def function_to_latex(f):
    print("Funktion:")
    display(Math(sp.latex(f,long_frac_ratio=2).replace('dstringdenniemandbenutzt',r'\Delta ')))
    print("Latex Code:")
    print(sp.latex(f,long_frac_ratio=2).replace('dstringdenniemandbenutzt',r'\Delta '))
    

In [15]:
#Hier soll eine Funktion entstehen, die eine Funktion auf einen Wert anwendet und gleichzeitig den Fehler ausrechnet
#Erstmal für Eingabe der Daten in Form von 2d Arrays

"""
Gibt pro Zeile den Wert und den absoluten Fehler der Daten bzgl. der gegebenen Funktion aus
Zusätzlich printet sie den Latex Code für die Formel, sowie die Fehlerformeln (Beim relativen Fehler unbedingt an die Multiplikation vom Ergebnis denken!)

Argumente:
function: sympy Funktion, in die die Messwerte eingesetzt werden sollen

params: Alle Parameter dieser Funktion als Array von Sympy-Zeichen

data: 2D-Array mit den Messdaten, sodass die Zeilen die Form haben: [Parameter 1, Fehler Parameter 1, Parameter 2,...]
Die Funktion wird zeilenweise angewandt. Wird kein Fehler für einen Parameter angenommen, kann diese Spalte entweder mit dem Wert 0 
an die Funktion gegeben werden oder ganz weggelassen werden. Dann muss allerdings der betreffende Parameter bei params_without_error angegeben werden.

params_without_error: Alle Parameter zu denen kein Fehler explizit in den Daten angegeben ist. Dieser wird auf 0 gesetzt und kommt dann
auch nicht in der Latex Form der Fehlerformel vor. Default: []

latex: Boolean der angibt, ob Funktion und Fehlerfunktion als Latex-Code ausgegeben werden sollen. Default: True
"""
def uncertainty(function,params, data, params_without_error=[],latex = True):
    #print(data)
    #Expand data to include the uncertainty 0 for values with no assigned uncertainty
    exp_data = np.zeros((data.shape[0],data.shape[1]+len(params_without_error)))
    i = 0      #läuft durch die Parameter
    j = 0      #läuft durch die expanded data
    z = 0      #läuft durch die eingegebene data
    while (i < len(params)):
        #print(exp_data)
        if (params[i] in params_without_error):
            exp_data[:,j] = data[:,z]
            i = i + 1
            j = j + 2
            z = z + 1
        else:
            exp_data[:,j] = data[:,z]
            exp_data[:,j+1] = data[:,z+1]
            i = i + 1
            j = j + 2
            z = z + 2
    #print(exp_data)
    #Create variable that stores parameters that have no assigned uncertainty    
    params_with_error = []
    j = 0
    for n in np.arange(0,len(params)):
        if not (params[n] in params_without_error):
            params_with_error.append(params[n])
            j = j + 1
    
    #Get the given function and error function as numpy functions
    f = sp.lambdify(params,function)
    error_abs, error_rel, parameters = errf(function,params)
    err_abs = sp.lambdify(np.concatenate(parameters),error_abs) #Achtung Argumentreihenfolge!!!! a, da, b, db, usw

    #Calculate the results for each row of data
    results = np.zeros((data.shape[0],2))
    for n in np.arange(0,data.shape[0]):
        results[n,:] = evaluate_with_error(f, err_abs, exp_data[n,:])
        #print(evaluate_with_error(f, err_abs, exp_data[n]))
    
    if (len(results) < 10):
        print("Results:")
        print(results)

    #Substitutes 0 for the uncertainty of the parameters without error, so it doesnt show up in the Latex Code
    for p in params_without_error:
        error_abs = error_abs.subs('dstringdenniemandbenutzt'+p.name,0)
    for p in params_without_error:
        error_rel = error_rel.subs('dstringdenniemandbenutzt'+p.name,0)
    if(latex == True):
        #Prints out the Latex Code for the Functions
        function = sp.simplify(function,symbols = params, rational= True)
        function = sp.separatevars(function)
        function_to_latex(function)
        print("Absoluter Fehler:")
        function_to_latex(error_abs)
        print("Relativer Fehler:")
        function_to_latex(error_rel)


    return(results)

In [16]:
"""
Macht das Selbe wie Uncertainty, akzeptiert aber die Daten in einer anderen Form.
data: Array, wobei jedes Element die Werte eines Parameters enthält (Reihenfolge [Wert-Parameter1, Fehler-Parameter1, Wert-Parameter2, etc.])
      Wert im Array kann entweder ein Array sein, oder nur ein einzelner Wert. Die Einschränkung ist, dass alle Werte in Array Form gleich lang sein müssen.
      Wird nur ein einzelner Wert angegeben, so ist dieser Parameter in allen Rechnungen konstant
"""
def uncertainty2(function, params,data, params_without_error = [],latex = True):
    length = 1
    for e in data:
        if(np.shape(np.array(e)) != ()):
            length = np.shape(np.array(e))[0]
            break
    data_2d = np.zeros((length,len(data)))
    for i in range (0,len(data)):
        data_2d[:,i] = data[i]
    return(uncertainty(function,params, data_2d,params_without_error = params_without_error,latex = latex))

# Beispiele

In [9]:
#Beispieleingabe für die Funktion "uncertainty" (Daten aus Jens' Fehlerrechner)
a,b,c = sp.symbols('a b c')
params = [a,b,c]
function = (a*b**2*sp.sqrt(c))
testd = np.array([[2.8,0.3,4.2,0.2,2.4,0.1]])
z = uncertainty(function,params, testd,params_without_error = [])

Results:
[[76.51775737 11.0842289 ]]
Funktion:


<IPython.core.display.Math object>

Latex Code:
a b^{2} \sqrt{c}
Absoluter Fehler:
Funktion:


<IPython.core.display.Math object>

Latex Code:
\frac{1}{2} \sqrt{\frac{b^{2}}{c} \left(a^{2} b^{2} \Delta c^{2} + 4 c^{2} \left(4 a^{2} \Delta b^{2} + b^{2} \Delta a^{2}\right)\right)}
Relativer Fehler:
Funktion:


<IPython.core.display.Math object>

Latex Code:
\frac{1}{2} \sqrt{\frac{\Delta c^{2}}{c^{2}} + \frac{16 \Delta b^{2}}{b^{2}} + \frac{4 \Delta a^{2}}{a^{2}}}


In [10]:
#Beispieleingabe für die Funktion uncertainty2 (Daten aus Versuch212)
#Berechnung der Viskosität nach H-P

#Daten aus dem Versuch
L=100e-3   #Länge in m
fL=0.05e-2
Rad=0.75e-3  #Radius in m
fRad=0.000005
p = np.array([5931.81208923])
fp = np.array([24.07782089])
dotV = 2.817496654193227e-08
fdotV = 9.431864027625913e-11

#Definition von Symbolen und Parametern
vp, vr, V, l ,pi= sp.symbols('p R \\dot{V} L \pi')
params = [vp,vr,V,l,pi]
function = (pi*vp*vr**4)/(l*8*V)

#Eingabe der Daten
werte = [p,fp,Rad,fRad,dotV,fdotV, L, fL,np.pi]
uncertainty2(function,params,werte,params_without_error = [pi])

Results:
[[0.26159468 0.00722965]]
Funktion:


<IPython.core.display.Math object>

Latex Code:
\frac{R^{4} \pi p}{8 L \dot{V}}
Absoluter Fehler:
Funktion:


<IPython.core.display.Math object>

Latex Code:
\frac{1}{8} \sqrt{\frac{R^{6}}{L^{4} \dot{V}^{4}} \left(L^{2} R^{2} \pi^{2} \Delta \dot{V}^{2} p^{2} + L^{2} \dot{V}^{2} \left(R^{2} \pi^{2} \Delta p^{2} + 16 \pi^{2} \Delta R^{2} p^{2}\right) + R^{2} \dot{V}^{2} \pi^{2} \Delta L^{2} p^{2}\right)}
Relativer Fehler:
Funktion:


<IPython.core.display.Math object>

Latex Code:
\sqrt{\frac{\Delta p^{2}}{p^{2}} + \frac{\Delta \dot{V}^{2}}{\dot{V}^{2}} + \frac{16 \Delta R^{2}}{R^{2}} + \frac{\Delta L^{2}}{L^{2}}}


array([[0.26159468, 0.00722965]])

# Random Stuff

In [11]:
data

array([(0,  11., 3.93), (1,  15., 3.8 ), (2,  45., 3.6 ), (3, 456., 4.5 ),
       (4,   8., 8.9 ), (5,  42., 5.3 ), (6,  93., 2.6 )],
      dtype=[('Zeit', '<i4'), ('Messwert1', '<f8'), ('Messwert2', '<f8')])

In [123]:
type(params[0])

sympy.core.symbol.Symbol

In [127]:
params_without_error = [a,b]
data = array
exp_data = np.zeros((data.shape[0],len(params)+len(params_without_error)))
i = 0
j = 0
while (i < len(params)):
    if (params[i] in params_without_error):
        exp_data[:,j] = data[:,i]
        i = i + 1
        j = j + 2
    else:
        exp_data[:,j] = data[:,i]
        i = i + 1
        j = j + 1

#params_with_error = np.zeros(len(params)-len(params_without_error),dtype ="sympy.core.symbol.Symbol")
params_with_error = []
j = 0
for n in np.arange(0,len(params)):
    if not (params[n] in params_without_error):
        params_with_error.append(params[n])
        j = j + 1


In [128]:
print(params_with_error)


[c]


In [120]:
params

[a, b, c]

In [111]:
0:len(params)

SyntaxError: illegal target for annotation (1861745115.py, line 1)

In [75]:
array = np.array(data.tolist())
array

array([[  0.  ,  11.  ,   3.93],
       [  1.  ,  15.  ,   3.8 ],
       [  2.  ,  45.  ,   3.6 ],
       [  3.  , 456.  ,   4.5 ],
       [  4.  ,   8.  ,   8.9 ],
       [  5.  ,  42.  ,   5.3 ],
       [  6.  ,  93.  ,   2.6 ]])

In [73]:
type(sarray)

numpy.ndarray

In [74]:
sarray[1]

IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [62]:
array[:,1]

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [159]:
function_to_latex(function)

Funktion:


<IPython.core.display.Math object>

Latex Code:
a b^{2} \sqrt{c}


In [13]:
display(Math(r'\Delta f='+sp.latex(test[0],long_frac_ratio=2).replace('d',r'\Delta ')))

<IPython.core.display.Math object>

In [22]:
np.concatenate(testvalues)[::2]

NameError: name 'testvalues' is not defined

In [23]:
x = [(1,4,5)]
#y = np.concatenate(x)
np.ndim(x)

2

In [40]:
testf = (1/2*a + b)
testparams =[a,b]
testvalues = (5,5,10,5)
#Um eine sympy Funktion zu evaluaten, sollte sie zuvor mit lambdify in eine numpy Funktion umgewandelt werden.
f = sp.lambdify(testparams,testf)
error_abs, error_rel, parameters = errf(testf,testparams)
err_abs = sp.lambdify(np.concatenate(parameters),error_abs) #Achtung Argumentreihenfolge!!!! a, da, b, db, usw

#def evaluate_with_error(f, err, values):
#    result = f(*values[:][0])
#    uncertainty = err(*np.concatenate(testvalues))
#    return(result, uncertainty)
    

In [41]:
np.concatenate(testvalues)

ValueError: zero-dimensional arrays cannot be concatenated

In [42]:
f(5)

TypeError: _lambdifygenerated() missing 1 required positional argument: 'b'

In [43]:
err= err_abs
evaluate_with_error(f,err,testvalues)

(12.5, 5.5901699437494745)

In [59]:
testvalues[:][0]

[5, 0.5]

In [47]:
testvalues

[(5, 0.5)]

In [11]:
a,b,c = sp.symbols('a b c')
#a = sp.symbols('a')
#b = sp.symbols('b')
#c = sp.symbols('c')

function = (a*b**2*sp.sqrt(c))


In [19]:
test[2]

[(a, da), (b, db), (c, dc)]

In [16]:
test =sp.simplify(errf(function, [a,b,c]))



In [7]:
test

(sqrt(b**2*(a**2*b**2*dc**2 + 4*c**2*(4*a**2*db**2 + b**2*da**2))/c)/2, sqrt(dc**2/c**2 + 16*db**2/b**2 + 4*da**2/a**2)/2)